# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a Random Forest Classifier for the Content Refresh lane. The task is a classification problem because the goal is to identify content items that should receive higher refresh priority. Random Forest is suitable because it can capture nonlinear relationships between search demand and performance signals without requiring strong assumptions about the relationship between the features and the decision. It also provides feature importance values that can help explain which observed signals the model relies on.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

df = pd.read_csv("content_refresh_anonymized.csv")

for col in ["search_volume", "impressions_90d", "clicks_90d"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["ctr_90d"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, np.nan)
)

df = df.dropna(
    subset=[
        "search_volume",
        "impressions_90d",
        "clicks_90d",
        "ctr_90d"
    ]
).copy()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 8534
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'ctr_90d']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a stratified random train-test split with 75% of the observations for training and 25% for testing. The same split is used for both the baseline and the Random Forest model so that the comparison is fair. Stratification keeps the proportion of priority and non-priority examples similar across the two sets. Because this starter dataset does not provide a reliable client grouping field for the modeling task, I will not claim that this split measures generalization to completely unseen clients.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr_90d"
]

X = df[feature_cols].copy()

threshold = df["impressions_90d"].median()

df["refresh_priority"] = (
    df["impressions_90d"] < threshold
).astype(int)

y = df["refresh_priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training positive rate:", y_train.mean())
print("Testing positive rate:", y_test.mean())

Training rows: 6400
Testing rows: 2134
Training positive rate: 0.49984375
Testing positive rate: 0.49953139643861294


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will compare the Random Forest against a simple baseline using the same test set and the same F1 metric. The baseline predicts the majority class for every test observation. The Random Forest receives the same historical features and is evaluated on exactly the same held-out observations. This makes the comparison directly interpretable.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_prediction = np.full(
    len(y_test),
    y_train.mode()[0]
)

baseline_f1 = f1_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

baseline_precision = precision_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

model_prediction = model.predict(X_test)

model_f1 = f1_score(
    y_test,
    model_prediction,
    zero_division=0
)

model_precision = precision_score(
    y_test,
    model_prediction,
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_prediction,
    zero_division=0
)

comparison = pd.DataFrame({
    "Model": [
        "Majority baseline",
        "Random Forest"
    ],
    "F1": [
        baseline_f1,
        model_f1
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ],
    "Recall": [
        baseline_recall,
        model_recall
    ]
})

display(comparison)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        model_prediction,
        digits=3,
        zero_division=0
    )
)

,Model,F1,Precision,Recall
0,Majority baseline,0.0,0.0,0.0
1,Random Forest,1.0,1.0,1.0



Classification report:
              precision    recall  f1-score   support

           0      1.000     1.000     1.000      1068
           1      1.000     1.000     1.000      1066

    accuracy                          1.000      2134
   macro avg      1.000     1.000     1.000      2134
weighted avg      1.000     1.000     1.000      2134



In [5]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
1,impressions_90d,0.728758
2,clicks_90d,0.186742
3,ctr_90d,0.083941
0,search_volume,0.000560


In [6]:
cm = confusion_matrix(
    y_test,
    model_prediction
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[1068    0]
 [   0 1066]]


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The error analysis focuses on false positives and false negatives rather than only the overall score. False positives are content items that the model recommends for refresh but are not in the defined priority class. False negatives are priority items that the model fails to identify. These errors matter because a false positive can spend review or refresh resources unnecessarily, while a false negative can cause a potentially useful refresh opportunity to be missed.

The feature importance values show which observed signals the Random Forest relies on most. These should be interpreted as model associations rather than causal effects. The model does not prove that changing one feature will improve search performance or that a refresh will cause better rankings.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
error_analysis = X_test.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = model_prediction

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) &
        (error_analysis["predicted"] == 0),

        (error_analysis["actual"] == 0) &
        (error_analysis["predicted"] == 1)
    ],
    [
        "FALSE_NEGATIVE",
        "FALSE_POSITIVE"
    ],
    default="CORRECT"
)

print("Error counts:")
print(error_analysis["error_type"].value_counts())

print("\nFalse negatives:")
display(
    error_analysis[
        error_analysis["error_type"] == "FALSE_NEGATIVE"
    ].head(10)
)

print("\nFalse positives:")
display(
    error_analysis[
        error_analysis["error_type"] == "FALSE_POSITIVE"
    ].head(10)
)

Error counts:
error_type
CORRECT    2134
Name: count, dtype: int64

False negatives:


,search_volume,impressions_90d,clicks_90d,ctr_90d,actual,predicted,error_type



False positives:


,search_volume,impressions_90d,clicks_90d,ctr_90d,actual,predicted,error_type


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.